In [1]:
import sys
from pathlib import Path

_here = Path().resolve()
for _candidate in [_here, _here.parent, _here.parent.parent]:
    if (_candidate / 'bughunt_env').is_dir():
        PROJECT_ROOT = _candidate
        break
else:
    raise RuntimeError('Could not find project root. Run Jupyter from inside the BugginHell folder.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

Project root: /Users/jagath-sajjan/BugginHell


In [3]:
import importlib, sys

# Remove any cached bughunt_env that pip/conda installed
for key in list(sys.modules.keys()):
    if 'bughunt' in key:
        del sys.modules[key]

# Confirm it's loading from your local folder
import bughunt_env
print("Loading from:", bughunt_env.__file__)

Loading from: None


In [2]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from bughunt_env import BugHuntEnv

ImportError: cannot import name 'BugHuntEnv' from 'bughunt_env' (unknown location)

In [ ]:
ACTION_COUNT = 5

def encode_obs(obs):
    text = " ".join([
        " ".join(obs.file_tree),
        obs.failing_test,
        obs.stderr,
        obs.last_tool_output,
        str(obs.steps_left),
    ]).lower()

    features = [
        obs.steps_left / 10,
        float("failed" in text),
        float("assert" in text),
        float("zerodivision" in text),
        float("calculate_total" in text),
        float("is_admin" in text),
        float("safe_divide" in text),
        float("normalize_name" in text),
        float("cart.py" in text),
        float("auth.py" in text),
        float("math_tools.py" in text),
        float("user_format.py" in text),
        float("tests/" in text),
    ]

    return torch.tensor(features, dtype=torch.float32)

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, input_dim=13, action_dim=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
import re as _re

def build_action(action_id, obs):
    """
    Convert action_id to env action tuple.
    Params are derived from observation signals — no hardcoded case lookup.
    """
    stderr = obs.stderr.lower()
    tool_out = obs.last_tool_output.lower()

    # Infer symbol from error text
    symbol_matches = _re.findall(r'\b([a-z_][a-z0-9_]{3,})\s*[\(=]', stderr + ' ' + tool_out)
    noise = {'assert', 'where', 'from', 'import', 'true', 'false', 'none', 'with', 'raise'}
    candidates = [s for s in symbol_matches if s not in noise]
    guessed_symbol = candidates[0] if candidates else obs.failing_test

    # Infer file from tool output
    file_match = _re.search(r'found in ([^\s]+\.py)', tool_out)
    caller_match = _re.search(r'from ([^\s]+\.py)', tool_out)
    src_files = [f for f in obs.file_tree if f.endswith('.py') and 'test' not in f]

    if file_match:
        guessed_file = file_match.group(1)
    elif caller_match:
        guessed_file = caller_match.group(1)
    elif src_files:
        guessed_file = src_files[0]
    else:
        guessed_file = obs.file_tree[0]

    # Infer line from tool output
    line_match = _re.search(r'line[:\s]+(\d+)', tool_out)
    if line_match:
        guessed_line = int(line_match.group(1))
    else:
        bug_keywords = ['range(len', '!=', '==', '- 1', '+ 1', 'return True', 'return False', 'None']
        guessed_line = 1
        for idx, line in enumerate(obs.last_tool_output.splitlines(), start=1):
            if any(kw in line for kw in bug_keywords):
                guessed_line = max(1, idx - 1)
                break

    if action_id == 0:
        return (0, {'path': guessed_file})
    if action_id == 1:
        return (1, {'name': obs.failing_test})
    if action_id == 2:
        return (2, {'name': guessed_symbol})
    if action_id == 3:
        return (3, {'fn': guessed_symbol})
    return (4, {'file': guessed_file, 'line': guessed_line})


In [ ]:
policy = PolicyNet()
optimizer = optim.Adam(policy.parameters(), lr=1e-3)

episode_rewards = []
successes = []

for episode in range(300):
    env = BugHuntEnv(seed=episode)
    obs, info = env.reset(seed=episode)

    log_probs = []
    rewards = []
    success = 0

    for step in range(env.max_steps):
        x = encode_obs(obs)
        logits = policy(x)
        dist = torch.distributions.Categorical(logits=logits)

        action_id = dist.sample()
        log_probs.append(dist.log_prob(action_id))

        action = build_action(action_id.item(), obs)
        obs, reward, terminated, truncated, info = env.step(action)

        rewards.append(reward)

        if terminated or truncated:
            if "correct_file=True, correct_line=True" in obs.last_tool_output:
                success = 1
            break

    total_reward = sum(rewards)
    episode_rewards.append(total_reward)
    successes.append(success)

    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + 0.99 * G
        returns.insert(0, G)

    returns = torch.tensor(returns, dtype=torch.float32)

    if len(returns) > 1:
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    loss = 0
    for log_prob, G in zip(log_probs, returns):
        loss -= log_prob * G

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if episode % 25 == 0:
        print(
            f"episode={episode}, reward={total_reward:.2f}, "
            f"success_rate_last_25={sum(successes[-25:]) / max(1, len(successes[-25:])):.2f}"
        )

In [ ]:
window = 25
success_rate = [
    sum(successes[max(0, i-window):i+1]) / len(successes[max(0, i-window):i+1])
    for i in range(len(successes))
]

plt.figure()
plt.plot(success_rate)
plt.xlabel("Episode")
plt.ylabel("Rolling Success Rate")
plt.title("BugginHell RL Agent Success Rate")
plt.show()

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

torch.save(policy.state_dict(), OUTPUT_DIR / "policy_net.pt")
print("Saved:", OUTPUT_DIR / "policy_net.pt")